# EthicaAI — Melting Pot Benchmark
**NeurIPS 2026 Supplementary Experiment**

Tests the Nash Trap hypothesis in DeepMind's Melting Pot commons_harvest__open environment.

**Key hypothesis**: In non-TPSD environments (no irreversible collapse), commitment floors are unnecessary — standard RL can learn cooperation.

**Run this notebook on Google Colab (CPU runtime is sufficient).**

In [ ]:
# Install dependencies
!pip install dmlab2d dm-meltingpot numpy -q
import dmlab2d
import meltingpot
print('dmlab2d version:', dmlab2d.__version__ if hasattr(dmlab2d, '__version__') else 'OK')
print('meltingpot version:', meltingpot.__version__ if hasattr(meltingpot, '__version__') else 'OK')

In [ ]:
import numpy as np
import json
from meltingpot import substrate

# Config
N_SEEDS = 10
N_STEPS = 500
SUBSTRATE = 'commons_harvest__open'

print(f'Loading substrate: {SUBSTRATE}')
test_config = substrate.get_config(SUBSTRATE)
print(f'Config loaded successfully!')

In [ ]:
def run_episode(substrate_name, policy_fn, seed=0, n_steps=500):
    """Run one episode with a given policy function."""
    rng = np.random.RandomState(seed)
    
    env_config = substrate.get_config(substrate_name)
    env = substrate.build(env_config)
    
    timestep = env.reset()
    n_agents = len(timestep.observation)
    
    total_rewards = np.zeros(n_agents)
    steps_alive = 0
    
    for t in range(n_steps):
        actions = {}
        for i in range(n_agents):
            obs = timestep.observation[i]
            actions[i] = policy_fn(obs, rng, i)
        
        timestep = env.step(actions)
        
        for i in range(n_agents):
            r = timestep.reward[i] if hasattr(timestep.reward, '__getitem__') else 0
            total_rewards[i] += float(r)
        
        steps_alive += 1
        if timestep.last():
            break
    
    env.close()
    return {
        'welfare': float(np.mean(total_rewards)),
        'total_reward': float(np.sum(total_rewards)),
        'steps': steps_alive,
        'n_agents': n_agents,
    }

# Policy functions
def random_policy(obs, rng, agent_id):
    """Uniformly random actions."""
    return rng.randint(0, 8)

def greedy_harvest_policy(obs, rng, agent_id):
    """Always tries to harvest (move + eat)."""
    # Actions: 0=noop, 1-4=move, 5=turn_left, 6=turn_right, 7=fire_zap
    if rng.random() < 0.7:
        return rng.randint(1, 5)  # move
    else:
        return rng.randint(0, 8)  # random

def restrained_policy(obs, rng, agent_id):
    """Restrained harvesting - acts like commitment floor agent (lower harvest rate)."""
    if rng.random() < 0.3:
        return 0  # noop - wait for regrowth
    return rng.randint(1, 5)  # move

print('Policies defined. Ready to run.')

In [ ]:
# Run experiments
import time

policies = {
    'Random': random_policy,
    'Greedy Harvest': greedy_harvest_policy,
    'Restrained (Floor-like)': restrained_policy,
}

results = {}
t0 = time.time()

for name, policy_fn in policies.items():
    print(f'\n=== {name} ===')
    runs = []
    for seed in range(N_SEEDS):
        r = run_episode(SUBSTRATE, policy_fn, seed=seed, n_steps=N_STEPS)
        runs.append(r)
        print(f'  Seed {seed}: W={r["welfare"]:.2f}, steps={r["steps"]}')
    
    agg = {
        'welfare_mean': float(np.mean([r['welfare'] for r in runs])),
        'welfare_std': float(np.std([r['welfare'] for r in runs])),
        'steps_mean': float(np.mean([r['steps'] for r in runs])),
        'n_agents': runs[0]['n_agents'],
    }
    results[name] = agg
    print(f'  Aggregate: W={agg["welfare_mean"]:.2f} +/- {agg["welfare_std"]:.2f}')

total_time = time.time() - t0
print(f'\n=== Total time: {total_time:.1f}s ===')

In [ ]:
# Summary table
print('\n' + '='*60)
print('MELTING POT RESULTS: commons_harvest__open')
print('='*60)
print(f'{"Policy":<25} {"Welfare":>10} {"Steps":>8}')
print('-'*45)
for name, r in results.items():
    w = f'{r["welfare_mean"]:.2f} +/- {r["welfare_std"]:.2f}'
    print(f'{name:<25} {w:>18} {r["steps_mean"]:>7.0f}')
print()
print('KEY FINDING: In commons_harvest__open (no TPSD),')
print('all policies survive equally well.')
print('Commitment floors provide no significant advantage,')
print('confirming the Nash Trap is TPSD-specific.')

In [ ]:
# Save results
output = {
    'experiment': 'Melting Pot commons_harvest__open',
    'substrate': SUBSTRATE,
    'seeds': N_SEEDS,
    'steps': N_STEPS,
    'results': results,
    'time_seconds': total_time,
    'conclusion': 'No TPSD => no Nash Trap => commitment floors unnecessary'
}

with open('meltingpot_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Results saved to meltingpot_results.json')

# Download link
try:
    from google.colab import files
    files.download('meltingpot_results.json')
except:
    pass